# QCNN Colab Training
Use this notebook on Google Colab to clone the QCNN project, install dependencies, preprocess data, and launch training on GPU instances such as A100.
Run each section in order and review the configuration cell before executing the pipeline.


## 1. Verify GPU runtime
Confirm that the notebook runtime has a GPU attached. Switch to an A100 runtime via the Runtime menu if needed.


In [ ]:
!nvidia-smi


## 2. Optional Google Drive mount
Mount Drive if you want to persist logs or checkpoints between sessions. Set the flag in the next cell to True before running it.


In [ ]:
from pathlib import Path

MOUNT_DRIVE = False  # Set True to keep outputs on Google Drive
DRIVE_ROOT = "/content/drive"

if MOUNT_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount(DRIVE_ROOT, force_remount=True)
    OUTPUT_BASE = Path(DRIVE_ROOT) / "MyDrive" / "QCNN_outputs"
else:
    OUTPUT_BASE = Path("/content/QCNN_outputs")

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
OUTPUT_BASE


## 3. Clone or refresh the QCNN repository
If the project directory already exists the cell will fetch the latest changes on the selected branch instead of cloning again.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/NikolaStarx/QCNN_project.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/QCNN_project")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "origin", BRANCH], check=True)

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")


## 4. Install Python dependencies
Use pip to install the required packages listed in requirements.txt. This may take a few minutes the first time you run the notebook.


In [ ]:
USE_AER_GPU = True   # Set False to keep the CPU-only Aer build
AER_VERSION = "0.17.1"

import subprocess
import sys

def run(cmd):
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])

if USE_AER_GPU:
    run([sys.executable, "-m", "pip", "uninstall", "-y", "qiskit-aer"])
    run([sys.executable, "-m", "pip", "install", f"qiskit-aer-gpu=={AER_VERSION}"])


## 5. Configure the experiment
Select the YAML config you want to run and extract key preprocessing parameters. Update CONFIG_PATH to point to a different file if required.


In [ ]:
from pathlib import Path
import yaml

CONFIG_PATH = Path("configs/mnist_amplitude_quick.yaml")
with open(CONFIG_PATH, "r", encoding="utf-8") as stream:
    CONFIG = yaml.safe_load(stream)

DATASET = CONFIG["data"]["dataset"].lower()
ENCODING = CONFIG["data"]["encoding"]
NUM_QUBITS = CONFIG["data"]["num_qubits"]

print(f"Using config: {CONFIG_PATH}")
print(f"Dataset: {DATASET}, encoding: {ENCODING}, num_qubits: {NUM_QUBITS}")


## 6. Preprocess data
Download the dataset if needed and prepare the amplitude encoded tensors that the training script expects.


In [ ]:
import subprocess
import sys

preprocess_cmd = [
    sys.executable,
    "scripts/preprocess.py",
    DATASET,
    ENCODING,
    "--num_qubits",
    str(NUM_QUBITS),
]

print("Running:", " ".join(preprocess_cmd))
subprocess.run(preprocess_cmd, check=True)


## 7. Train the QCNN model
Launch the training loop using the selected configuration. Outputs appear below and checkpoints plus logs are saved under the project directory.


In [ ]:
import subprocess
import sys

train_cmd = [sys.executable, "train.py", "--config", str(CONFIG_PATH)]

print("Running:", " ".join(train_cmd))
subprocess.run(train_cmd, check=True)


## 8. Optional: package outputs
Copy logs and checkpoints into the output folder defined earlier and create a zip archive for download or Drive storage.


In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

run_dir = OUTPUT_BASE / f"qcnn_run_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
run_dir.mkdir(parents=True, exist_ok=True)

for name in ["logs", "checkpoints"]:
    src = Path(name)
    if src.exists():
        dest = run_dir / name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(src, dest)

archive_path = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
archive_path
